In [83]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.lines import Line2D
import math
import matplotlib.cm as cm
import matplotlib as mpl
from Testing.RTLO import *
from Functions.RLS import *
from Functions.Utils import *
from Functions.Graphs import *
from Functions.TedaGraphs import *

dir_hi = r"C:\Artigo_J3C_V2\VMD10\HI"
dir_rs = r"C:\Artigo_J3C_V2\VMD10\RS6_def\S2_A1_Q10_W10_RS_raw\RS1"
brng = 'Bearing1_1.csv'
Mp=[1.9249999999999998, 10, 8, True]
Mp2 = [1e-7,	0.1,	0.01,	1e-7,	18,	3,	24]
m, nRS, st, drop = Mp
D,N1, N2, N3,nHI, nR,tau = Mp2

In [84]:
def euclidian_dist(x1,x2):
    return np.linalg.norm(x1 - x2,ord=None)

class DataCloud:
	N=0
	def __init__(self,x,ID,nI=1,nR=1,nO=1,ηS=[1,1,1],tau=1,decay=1):
		self.ID = ID
		self.track = [ID]
		self.merged = False
		self.merge = None
		self.n=1
		self.mean=x
		self.meant=np.array(x).dot(np.array(x))
		self.variance=0
		self.pertinency=1
		self.tipicality=1e-12
		self.nS = len(x)
		self.nI = nI
		self.nR = nR
		self.nO = nO
		self.N1 = ηS[0]
		self.N2 = ηS[1]
		self.N3 = ηS[2]
		self.decay = decay
		self.tau = tau
		self.rnn = RTLO(self.nI, self.nR, self.nO,
					[self.N1, self.N2, self.N3], self.tau, self.decay)
		self.x = [x]
		self.t = []
		self.R = 0
		self.xI = x
		self.xF = x
		self.Rmax = 0
		self.specificity = 0
		self.coverage = 0
		self.cardinality = 1
		self.v = 0
		self.Dmax = -np.inf
		
	def addDataClaud(self,x):
		self.n=2
		self.mean=(self.mean+x)/2
		self.meant=((self.meant)/2) + (x.dot(x))/2
		self.variance=self.meant-self.mean.dot(self.mean)

	def updateDataCloud(self,n,mean,meant,variance,tipicality):
		self.n=n
		self.mean=mean
		self.meant=meant
		self.variance=variance
		self.tipicality=tipicality
		self.cardinality=self.cardinality + 1

In [ ]:
class AutoCloud:
	def __init__(self,m,nS=1,nI=1,nR=1,nO=1,ηS=[0.1,0.1,0.1],
			  tau=1,decay=1,eol=0,fator=1,st=0,end=0,ep=0.1,wta=False):
		
		self.ep = ep
		self.st = st
		self.end = end
		self.nS = nS
		self.nI = nI
		self.nR = nR
		self.nO = nO
		self.N1 = ηS[0]
		self.N2 = ηS[1]
		self.N3 = ηS[2]
		self.tau = tau
		self.eol = eol
		self.decay = decay
		self.fator = fator
		self.g = 1
		self.gCreated = 0
		self.c= np.array([])
		self.alfa= np.array([0.0],dtype=float)
		self.intersection = np.zeros((1,1),dtype=int)
		self.listIntersection = np.zeros((1),dtype=int)
		self.matrixIntersection = np.zeros((1,1),dtype=int)
		self.relevanceList = np.zeros((1),dtype=int)
		self.classIndex = []
		self.k=1
		self.m = m
		self.cloud_activation = []
		self.cloud_activation2 = []
		self.cloud_activation3 = []
		self.aux = np.array([])
		self.DSIs = [np.array([]) for i in range(nS)]
		self.OffineGrnls = None

		self.eolX = 0
		self.HI = np.array([])
		self.DSI = np.array([])
		self.eolDSI = 0
		self.HIp = np.array([])
		self.cycleP=np.array([])

		self.rulL = np.array([])
		self.rulP = np.array([])
		self.rulU = np.array([])
		self.rulR = np.array([])
		self.rulR_Train = np.array([])

		self.rls = RLS_LogarithmicRegressor(0.9,10000)
		self.pError = []
		self.rError = []
		self.win_all = wta
		self.ChangePoint = []
		self.xR = 0
		self.xF = 0
		self.Dmax = 0
	
	def CreateCloud(self,x):
		self.gCreated = self.gCreated + 1
		cloud = DataCloud(x,self.gCreated,self.nI,self.nR,self.nO,
					[self.N1,self.N2,self.N3],self.tau,self.decay)
		self.c = np.append(self.c,cloud)

	def add_rulR2(self,n):
		self.rulR2 = np.array([n])

	def adapt(self,y,z):
		self.HI = np.append(self.HI,z[-1])

		if self.k >5 and self.HI[-1] < self.eol and self.eolX==0:
			self.eolX=self.cycleP[-1]-1
		wMax = -np.inf
		tS = sum([cloud.tipicality for cloud in self.c])
		wS = np.array([cloud.tipicality for cloud in self.c])/tS
		for w,cloud in zip(wS,self.c):
			if w > wMax:
				wMax = w
				Best_cloud = cloud
		if self.win_all:
			Best_cloud.rnn.fit(y,z)
		if not self.win_all:
			for i,cloud in enumerate(self.c):
				cloud.rnn.fit(y,z)

	def predict(self,y):
		wSum = sum([cloud.tipicality for cloud in self.c])
		ws = np.array([cloud.tipicality/wSum for cloud in self.c]).reshape(-1,1)
		p = np.array([cloud.rnn.PredSingle(y) for cloud in self.c])
		p1 = (p*ws)
		p1 = sum([p1[i][-1] for i in range(len(p))])
		if self.win_all:
			p1 = p[np.argmax(ws)][-1]
		return p1
	
	def predict2(self,y):
		wSum = sum([cloud.tipicality for cloud in self.c])
		ws = np.array([cloud.tipicality/wSum for cloud in self.c]).reshape(-1,1)
		p = np.array([cloud.rnn.PredSingle(y) for cloud in self.c])
		p1 = (p*ws)
		p1 = sum([p1[i][-1] for i in range(len(p))])
		if self.win_all:
			p1 = p[np.argmax(ws)][-1]
		return p1
	
	def restore_rnn(self):
		for cloud in self.c:
			cloud.rnn.restore()

	def RUL_single(self,X,Scut=None,wta=False):
		if wta: self.win_all = True
		if Scut is None: Scut = 160
		pP,pU,pL,rulP,rulU,rulL = [0 for i in range(6)]
		xP,xU_max,xU_min,xL_min,xL_max = [X.copy() for i in range(5)]
		pP = self.predict(xP)*self.fator
		eR = np.abs(self.HI[-1]-pP) 
		self.HIp = np.append(self.HIp,pP)
		vL,vU,vP=[],[],[]

		if pP>0:
			self.rls.update(np.abs(pP), eR)
			eP = self.rls.predict(np.abs(pP))
		self.restore_rnn()
		
		while xP[-1]>self.eol:
			pP = self.predict(xP)*self.fator
			xP = np.delete(np.append(xP,pP),0)
			rulP=rulP+1
			if rulP == Scut: 
				rulP = 0
				break

		self.restore_rnn()
		self.rulP = np.append(self.rulP,rulP)
		self.rulL = np.append(self.rulL,rulL)
		self.rulU = np.append(self.rulU,rulU)

		return
	def RUL_uncertainty(self,X):
		eP,eL,eU=0,0,0
		pP,pU,pL,rulP,rulU,rulL = [0 for i in range(6)]
		xP,xU_max,xU_min,xL_min,xL_max = [X.copy() for i in range(5)]
		pP = self.predict(xP)*self.fator
		eR = np.abs(self.HI[-1]-pP) 
		self.HIp = np.append(self.HIp,pP)
		vL,vU,vP=[],[],[]

		if pP>0:
			self.rls.update(np.abs(pP), eR)
			eP = self.rls.predict(np.abs(pP))

		self.pError.append(eP)
		self.rError.append(eR)

		self.restore_rnn()

		while xP[-1]>self.eol:
			pP = self.predict(xP)*self.fator
			#if self.k ==2: print('pP:',pP)
			xP = np.delete(np.append(xP,pP),0)
			rulP=rulP+1
			if rulP ==160: break
			vP.append(pP)
		self.restore_rnn()

		while xL_min[-1] > self.eol:
			#print(xL_min[:3],xL_max[:3])
			pL1 = self.predict(xL_min)*self.fator
			pL2 = self.predict2(xL_max)*self.fator
			eL1 = abs(self.rls.predict(np.abs(pL1)))
			eL2 = abs(self.rls.predict(np.abs(pL2)))
			pL_max = max([(pL1-eL1*self.ep),(pL1+eL1*self.ep),(pL2-eL2*self.ep),(pL2+eL2*self.ep),
				 (pL1-eL2*self.ep),(pL1+eL2*self.ep),(pL2-eL1*self.ep),(pL2+eL1*self.ep)])
			pL_min = min([(pL1-eL1*self.ep),(pL1+eL1*self.ep),(pL2-eL2*self.ep),(pL2+eL2*self.ep),
				 (pL1-eL2*self.ep),(pL1+eL2*self.ep),(pL2-eL1*self.ep),(pL2+eL1*self.ep)])
			xL_min = np.delete(np.append(xL_min,pL_min),0)  
			xL_max = np.delete(np.append(xL_max,pL_max),0)  
			rulL=rulL+1
			if rulL ==160: break
		#if self.k==50: print('pL:',(pL1-eL1*self.ep),(pL1+eL1*self.ep),(pL2-eL2*self.ep),(pL2+eL2*self.ep))
		#if self.k==50: print('eL:',eL1,eL2)

		self.restore_rnn()

		while xU_max[-1] > self.eol:
			
			pU1 = self.predict(xU_min)*self.fator
			pU2 = self.predict2(xU_max)*self.fator
			eU1 = abs(self.rls.predict(np.abs(pU1)))
			eU2 = abs(self.rls.predict(np.abs(pU2)))
			pU_max = max([(pU1-eU1*self.ep),(pU1+eU1*self.ep),(pU2-eU2*self.ep),(pU2+eU2*self.ep),
				 #(pU1-eU2*self.ep),(pU1+eU2*self.ep),(pU2-eU1*self.ep),(pU2+eU1*self.ep)
				 ])
			pU_min = min([(pU1-eU1*self.ep),(pU1+eU1*self.ep),(pU2-eU2*self.ep),(pU2+eU2*self.ep),
				 #(pU1-eU2*self.ep),(pU1+eU2*self.ep),(pU2-eU1*self.ep),(pU2+eU1*self.ep)
				 ])
			print(pU1,pU_max,pU_min)
			xU_min = np.delete(np.append(xU_min,pU_min),0)  
			xU_max = np.delete(np.append(xU_max,pU_max),0)  
			rulU=rulU+1
			if rulU ==160: break
		self.restore_rnn()

		self.rulP = np.append(self.rulP,rulP)
		self.rulL = np.append(self.rulL,rulL)
		self.rulU = np.append(self.rulU,rulU)
		return 
	
	def mergeClouds(self,x):
		i=0
		while(i<len(self.listIntersection)-1):
			merge=False
			j=i+1
			while(j<len(self.listIntersection)):
				if(self.listIntersection[i] == 1 and self.listIntersection[j] == 1):
					self.matrixIntersection[i,j] = self.matrixIntersection[i,j] + 1
				idI = self.c[i].ID
				idJ = self.c[j].ID
				meanI = self.c[i].mean
				meanJ = self.c[j].mean
				meantI = self.c[i].meant
				meantJ = self.c[j].meant
				nI = self.c[i].n
				nJ = self.c[j].n
				tipicalityI = self.c[i].tipicality
				tipicalityJ = self.c[j].tipicality
				trackI = self.c[i].track
				trackJ = self.c[j].track
				varianceI = self.c[i].variance
				varianceJ = self.c[j].variance
				winI = self.c[i].rnn.wIN
				winJ = self.c[j].rnn.wIN
				wrecI = self.c[i].rnn.wHS
				wrecJ = self.c[j].rnn.wHS
				woutI = self.c[i].rnn.wOUT
				woutJ = self.c[j].rnn.wOUT
				hiI = self.c[i].rnn.hI
				hiJ = self.c[j].rnn.hI
				nIntersc = self.matrixIntersection[i,j]

				if (nIntersc > (nI - nIntersc) or nIntersc > (nJ - nIntersc)):
					rI = np.max(self.c[i].R)
					rJ = np.max(self.c[j].R)
					if rI >= rJ: radius = rI
					else: radius = rJ
					merge = True
					self.gCreated = self.gCreated + 1
					n = nI + nJ - nIntersc
					mean = ((nI * meanI) + (nJ * meanJ))/(nI + nJ)
					meant = ((nI * meantI) + (nJ * meantJ))/(nI + nJ)
					variance = ((nI - 1) * varianceI + (nJ - 1) * varianceJ)/(nI + nJ - 2)
					tipicality = ((nI*tipicalityI)+(nJ*tipicalityJ))/(nI + nJ)
					wIN = ((winI*tipicalityI)+(winJ*tipicalityJ))/(tipicalityI+tipicalityJ)
					wHS = ((wrecI*tipicalityI)+(wrecJ*tipicalityJ))/(tipicalityI+tipicalityJ)
					wOUT = ((woutI*tipicalityI)+(woutJ*tipicalityJ))/(tipicalityI+tipicalityJ)
					hI = ((hiI*tipicalityI)+(hiJ*tipicalityJ))/(tipicalityI+tipicalityJ)

					newCloud = DataCloud(x,self.gCreated,self.nR,self.nO,
									[self.N1,self.N2,self.N3],self.tau,self.decay)
					for id in trackI:
						newCloud.track.append(id)
					for id in trackJ:
						newCloud.track.append(id)
						
					newCloud.updateDataCloud(n,mean,meant,variance,tipicality)
					newCloud.R = radius
					
					x = self.c[i].x + self.c[j].x
					t = self.c[i].t + self.c[j].t
					mat = np.array(list(zip(t,x)), dtype=object)
					col = mat[:, 0].astype(int) 
					_, index = np.unique(col, return_index=True)
					result = mat[np.sort(index)]
					t = result[:, 0].tolist()
					x = result[:, 1].tolist()
					newCloud.x = x
					newCloud.t = t
					dist = 0
					if euclidian_dist(self.c[i].xI,self.c[i].xF) > dist:
						dist = euclidian_dist(self.c[i].xI,self.c[i].xF)
						newCloud.xI = self.c[i].xI
						newCloud.xF = self.c[i].xF
					elif euclidian_dist(self.c[j].xI,self.c[j].xF) > dist:
						dist = euclidian_dist(self.c[j].xI,self.c[j].xF)
						newCloud.xI = self.c[j].xI
						newCloud.xF = self.c[j].xF
					elif euclidian_dist(self.c[i].xI,self.c[j].xF) > dist:
						dist = euclidian_dist(self.c[i].xI,self.c[j].xF)
						newCloud.xI = self.c[i].xI
						newCloud.xF = self.c[j].xF
					elif euclidian_dist(self.c[j].xI,self.c[i].xF) > dist:
						dist = euclidian_dist(self.c[j].xI,self.c[i].xF)
						newCloud.xI = self.c[j].xI
						newCloud.xF = self.c[i].xF
					newCloud.rnn.wIN = wIN
					newCloud.rnn.wHS = wHS
					newCloud.rnn.wOUT = wOUT
					newCloud.rnn.hI = hI
					newCloud.merge = f'G{self.gCreated}: G{idI}+G{idJ}'
					self.aux = np.append(self.aux,newCloud.ID)

					self.listIntersection = np.concatenate((self.listIntersection[0 : i], np.array([1]), self.listIntersection[i + 1 : j],self.listIntersection[j + 1 : np.size(self.listIntersection)]),axis=None)
					self.c = np.concatenate((self.c[0 : i ], np.array([newCloud]), self.c[i + 1 : j],self.c[j + 1 : np.size(self.c)]),axis=None)
					M0 = self.matrixIntersection
					M1=np.concatenate((M0[0 : i , :],np.zeros((1,len(M0))),M0[i + 1 : j, :],M0[j + 1 : len(M0), :]))
					M1=np.concatenate((M1[:, 0 : i ],np.zeros((len(M1),1)),M1[:, i+1 : j],M1[:, j+1 : len(M0)]),axis=1)
					col = (M0[:, i] + M0[:, j])*(M0[: , i]*M0[:, j] != 0)
					col = np.concatenate((col[0 : j], col[j + 1 : np.size(col)]))
					lin = (M0[i, :]+M0[j, :])*(M0[i, :]*M0[j, :] != 0)
					lin = np.concatenate((lin[ 0 : j], lin[j + 1 : np.size(lin)]))
					M1[:,i]=col
					M1[i,:]=lin
					M1[i, i + 1 : j] = M0[i, i + 1 : j] + M0[i + 1 : j, j].T;   
					self.matrixIntersection = M1
				j += 1
			if(merge): i = 0
			else: i += 1

	def run(self,X):
		self.aux = np.array([])
		self.aux2 = np.array([])
		if self.k == 1: self.xR = X
		self.listIntersection = np.zeros((np.size(self.c)),dtype=int)
		if self.k==1:
			self.CreateCloud(X)
			self.c[0].t.append(self.k)
			self.aux = np.append(self.aux,1)
			self.aux2 = np.append(self.aux,self.c[0].track)

		elif self.k==2:
			self.c[0].addDataClaud(X)
			self.c[0].x.append(X)
			self.c[0].t.append(self.k)
			v = self.c[0].variance
			n = self.c[0].n
			R = math.sqrt((((self.m**2)+1)*n*v/n)-v)
			self.c[0].R = R
			self.c[0].cardinality = self.c[0].cardinality+1
			self.aux = np.append(self.aux,1)
			self.aux2 = np.append(self.aux,self.c[0].track)

		elif self.k>=3:
			i=0
			createCloud = True
			self.alfa = np.zeros((np.size(self.c)),dtype=float)
			for cloud in self.c:
				n= cloud.n +1
				mean = ((n-1)/n)*cloud.mean + (1/n)*X
				meant = ((n-1)/n) * cloud.meant + (X.dot(X))/n
				variance=meant-mean.dot(mean)
				eccentricity = (1/n)+((mean-X).T.dot(mean-X))/(n*variance)
				typicality = 1 - (eccentricity-(1e-12))
				norm_eccentricity = eccentricity/2
				norm_typicality = typicality/(self.k-2)
				cloud.eccAn = eccentricity
				R = math.sqrt((((self.m**2)+1)*n*variance/n)-variance)
				if(norm_eccentricity<=(self.m**2 +1)/(2*n)):
					createCloud= False
					cloud.updateDataCloud(n,mean,meant,variance,typicality)
					self.alfa[i] = norm_typicality
					self.listIntersection.itemset(i,1)
					cloud.x.append(X)
					cloud.t.append(self.k)
					cloud.xF = X
					cloud.R = R
					self.aux = np.append(self.aux,cloud.ID)
					self.aux2 = np.append(self.aux2,cloud.track)
				else:
					self.alfa[i] = 0
					self.listIntersection.itemset(i,0)
				i+=1

			if(createCloud):
				self.CreateCloud(X)
				self.listIntersection = np.insert(self.listIntersection,i,1)
				self.matrixIntersection = np.pad(self.matrixIntersection, ((0,1),(0,1)), 'constant', constant_values=(0)) 
				self.c[-1].t.append(self.k)
				self.c[-1].R = 0
				self.aux = np.append(self.aux,self.c[-1].ID)
				self.aux2 = np.append(self.aux2,self.c[-1].track)
			self.mergeClouds(X)

		if self.Dmax < euclidian_dist(x1=self.xR, x2=X):
			self.Dmax = euclidian_dist(x1=self.xR, x2=X)
			self.xF = X

		self.cycleP = np.append(self.cycleP,self.st+self.k-1)
		self.rulR = np.flip(self.cycleP-(self.st-1))
		self.rulR_Train = np.append(self.rulR_Train,self.end-self.k)
		self.k=self.k+1

In [96]:
init = (st+nRS)
df_RS = (pd.read_csv(os.path.join(dir_rs,brng)).abs()).iloc[:,:-1]
df_HI = (pd.read_csv(os.path.join(dir_hi,brng)).abs()).iloc[:,-1:]
if drop: df_RS = df_RS.drop(columns='Y')
RS = process_RS(df_RS,nRS)
HI = process_HI(df_HI,nHI=nHI,init=init)
xS,yS,zS = RS[st:-2], HI[:-1], HI[1:]

teda=AutoCloud(m=m, nS=len(xS[0]), nI=len(yS[0]), nR=nR, nO=len(yS[0]), 
                            ηS= [N1, N2, N3], tau=tau, decay=D,eol=0.2,
                            fator=1,st=init,end=len(df_RS)-init,wta=True,ep=0.12) 

for j,_ in enumerate(xS):
    x,y,z= xS[j],yS[j],zS[j]
    teda.run(x)
    teda.adapt(y,z)
    #teda.RUL_single(y)
    teda.RUL_uncertainty(y)

#plot_2series(x1=teda.cycleP,x2=teda.cycleP,y1=teda.rulR,y2=teda.rulP, title=brng,s1='True',s2='Pred')
#plot_DSI2(teda,ftrs=2,title='DSI - Granulation',ncol=2)

-0.027310737271387418 -0.027310737271387418 -0.0401065514692512
-0.040881780053100496 -0.040881780053100496 -0.05297860153504649
-0.054362670858590215 -0.054362670858590215 -0.0656756804770284
-0.06736315571555401 -0.06736315571555401 -0.0778171853515701
-0.07944752800316768 -0.07944752800316768 -0.08897967977726222
-0.0901469114075162 -0.0901469114075162 -0.0987091180501212
-0.09897546550905256 -0.09897546550905256 -0.10653704591619409
-0.10544707815473756 -0.10544707815473756 -0.11199603319385294
-0.10909493413203494 -0.10909493413203494 -0.1146390773673171
-0.10949021298650392 -0.10949021298650392 -0.1140574279491091
-0.1062601534669754 -0.1062601534669754 -0.10989804135562577
-0.09910406519877893 -0.09910406519877893 -0.10187903618233848
-0.08780595615919087 -0.08780595615919087 -0.08980143715043987
-0.0722447110791548 -0.0722447110791548 -0.07355918537824097
-0.05239990015660162 -0.05239990015660162 -0.05314438059899988
-0.028353987073297518 -0.028353987073297518 -0.02864917011006

In [94]:
x = teda.cycleP
yL,yP,yU,yR = teda.rulL,teda.rulP,teda.rulU,teda.rulR
plot_series(series=[[x,x,x,x],[yL,yP,yU,yR]],names=['L','P','U','R'],show=False)